## Heatmaps

In [ ]:
import json
import numpy as np

from PIL import Image as PImage

from Museum import Museum
from utils.image_utils import heatmap_image, mask_image

NCLUSTERS = 9
TCLUSTERS = "umap"

In [ ]:
with open("./metadata/json/20260801_clusters.json", "r", encoding="utf8") as ifp:
  cdata = json.load(ifp)[f"{NCLUSTERS}"][TCLUSTERS]

descriptions_gemma3 = cdata["clusters"]["descriptions"]["gemma3"]["en"]
descriptions_siglip2 = cdata["clusters"]["descriptions"]["siglip2"]["en"]

with open("./metadata/json/20260801_activations.json", "r", encoding="utf8") as ifp:
  adata = json.load(ifp)

In [ ]:
iid = "Q61885913"
img = PImage.open(f"../../imgs/arts/500/{iid}.jpg")
img_np = np.array(img)

similarity_map = adata[iid]
similarity_map_np = np.array(similarity_map)
# similarity_map_np[similarity_map_np == similarity_map_np.max()] -= 2*similarity_map_np.std()
# similarity_map_np[similarity_map_np == similarity_map_np.max()] -= 2*similarity_map_np.std()
# similarity_map_np[similarity_map_np == similarity_map_np.max()] -= 2*similarity_map_np.std()

masked_img = mask_image(img, similarity_map_np)
heatmap_img = heatmap_image(similarity_map_np, size=img.size)

print(iid, descriptions_gemma3[cdata["images"][iid]["cluster"]])
display(img)
display(masked_img)
display(PImage.blend(img, heatmap_img, 0.65))

### Heatmap Tests

In [ ]:
!wget -O cat.jpg "https://images.thdstatic.com/productImages/8d8081f6904644ac94b2fba803425184/svn/cat-beds-sp-db1298bl-e1_600.jpg"
!wget -O cats.jpg "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcTxF9MeQRLhK5MzPioeN0RAhkuWrcUP7sE_nYPfLhrkgg&s=10"
!wget -O penguins.jpg "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQcm3Dg8Osl6Q5Osb47Fm288c2aC9UIm5mG5xz92SPcbQ&s=10"
!wget -O beatles.jpg "https://ichef.bbci.co.uk/news/480/cpsprodpb/3970/production/_108240741_beatles-abbeyroad-square-reuters-applecorps.jpg"

In [ ]:
import bisect
import json
import numpy as np

from PIL import Image as PImage

from Museum import Museum
from models.SigLip2 import SigLip2
from params.collections import MUSEUMS
from utils.image_utils import heatmap_image, mask_image, scale_2d_array

NCLUSTERS = 9
TCLUSTERS = "umap"

msl = SigLip2()

In [ ]:
museum_info = { k:v for k,v in MUSEUMS.items() if k != "file" }

for name,info in museum_info.items():
  print("combine:", name)
  Museum.combine_data(info)

embedding_data = Museum.combine_all_data(museum_info, "embeddings")
print(len(embedding_data), "works")

In [ ]:
with open("./metadata/json/20260801_clusters.json", "r", encoding="utf8") as ifp:
  cdata = json.load(ifp)[f"{NCLUSTERS}"][TCLUSTERS]

descriptions_gemma3 = cdata["clusters"]["descriptions"]["gemma3"]["en"]
descriptions_siglip2 = cdata["clusters"]["descriptions"]["siglip2"]["en"]

In [ ]:
def min_dist(iid):
  return min(cdata["images"][iid]["distances"])

by_clust_dist = [[] for _ in range(NCLUSTERS)]

In [ ]:
import matplotlib.cm as cm

from scipy.interpolate import RBFInterpolator
from scipy.ndimage import median_filter

def heatmap_image_rbf(data, size, cmap="inferno", kernel="thin_plate_spline"):
  xs = np.linspace(start=0, stop=size[0], num=data.shape[0])
  ys = np.linspace(start=0, stop=size[1], num=data.shape[0])
  ps = np.array([[x,y] for y in ys for x in xs])
  vs = data.reshape(-1)
  rbf = RBFInterpolator(ps, vs, kernel=kernel)
  x1 = np.arange(0, size[0])
  x2 = np.arange(0, size[1])
  xgrid = np.asarray(np.meshgrid(x1, x2, indexing="xy"))
  xflat = xgrid.reshape(2, -1).T
  yflat = rbf(xflat)
  ygrid = yflat.reshape(size[1], size[0])
  map_fun_np = np.vectorize(cm.get_cmap(cmap))
  rgba_np = 255 * np.stack(map_fun_np(ygrid)[:3], axis=-1)
  return PImage.fromarray(rgba_np.astype(np.uint8))

def std_filter(data, size, std=3):
  median = median_filter(data, size=size)
  diff = data - median
  threshold = std * np.std(diff)
  filtered = data.copy()
  mask = (diff > threshold) | (diff < -threshold)
  filtered[mask] = median[mask]
  return filtered

In [ ]:
# TEST IMAGES
img = PImage.open("./beatles.jpg")
img_np = np.array(img)

labels = ["person", "car", "cat", "penguin", "tree", "street"]
text_embs = msl.get_text_embedding(labels)

similarity_map_np = msl.get_gradient_activation_map(img, text_embs[:1], label_idx=None)
# similarity_map = ((1e6 * similarity_map_np).astype(int).astype(float) / 1e6).tolist()

heatmap_img = heatmap_image(similarity_map_np, size=img.size, sampling=PImage.Resampling.BILINEAR)
overlay_img = PImage.blend(img, heatmap_img.resize(img.size), 0.65)

# display(img)
display(mask_image(img, similarity_map_np))
display(overlay_img)

In [ ]:
# CLUSTER = 0 # iidx = 10, 24, 31, 32
# CLUSTER = 1 # iidx = 0, 2, 5, 8, 24, 26, 27, 28, 31, 63, 64, 104, 105
# CLUSTER = 2 # iidx = 2, 10, 22, 31
# CLUSTER = 3 # iidx = 1, 2, 15, 21, 26, 32, 153
# CLUSTER = 4 # iidx = 1, 5, 32, 33, 39, 40
# CLUSTER = 5 # iidx = 0, 10, 48, 50, 67, 69, 82
# CLUSTER = 6 # iidx = 10, 31, 37, 38, 48
CLUSTER = 7 # iidx = 9, 25, 35, 77
# CLUSTER = 8 # iidx = 3, 5, 12, 63

cdescsg, cdescss, cimgs = descriptions_gemma3[CLUSTER], descriptions_siglip2[CLUSTER], by_clust_dist[CLUSTER]
labels = cdescsg[:3]
print(labels)
iidx = 64

img = PImage.open(f"./herbario-media/imgs/arts/500/{cimgs[iidx]}.jpg")
img_np = np.array(img)

similarity_map_np = msl.get_gradient_activation_map(img, labels)
# similarity_map_np[similarity_map_np == similarity_map_np.max()] -= 2*similarity_map_np.std()
# similarity_map_np[similarity_map_np == similarity_map_np.max()] -= 2*similarity_map_np.std()
# similarity_map_np[similarity_map_np == similarity_map_np.max()] -= 2*similarity_map_np.std()
# similarity_map = ((1e6 * similarity_map_np).astype(int).astype(float) / 1e6).tolist()

masked_img = mask_image(img, similarity_map_np)
heatmap_img = heatmap_image(similarity_map_np, size=img.size)

print(cimgs[iidx], labels)
display(img)
display(masked_img)
display(PImage.blend(img, heatmap_img, 0.65))

In [ ]:
similarity_map_np_mean = similarity_map_np.mean()
similarity_map_np_std = similarity_map_np.std()
mp1s = similarity_map_np_mean + similarity_map_np_std
mp2s = similarity_map_np_mean + 2*similarity_map_np_std

similarity_map_npc = similarity_map_np.copy()
similarity_map_npc[similarity_map_np > mp1s] = 0.75
similarity_map_npc[similarity_map_np > mp2s] = 0.9
# similarity_map_npc[similarity_map_np < similarity_map_np_mean] = 0.1

masked_img = mask_image(img, similarity_map_npc)
heatmap_img = heatmap_image(similarity_map_npc, size=img.size)

print(cimgs[iidx], labels)
display(img)
display(masked_img)
display(PImage.blend(img, heatmap_img, 0.65))

In [ ]:
clust_rep_k = 16

def get_em_row(row):
  return [embedding_data[iid]["siglip2"] for iid in row]

for iid,v in cdata["images"].items():
  icluster = v["cluster"]
  bisect.insort(by_clust_dist[icluster], iid, key=min_dist)

by_clust_dist_top_k = np.array([x[:clust_rep_k] for x in by_clust_dist])
clust_embeddings_top_k = np.apply_along_axis(get_em_row, 1, by_clust_dist_top_k)
avg_cluster_embedding = clust_embeddings_top_k.mean(axis=1)

In [ ]:
# DOES NOT WORK AT ALL !!!
# PROBABLY BECAUSE CLUSTERS WERE CALCULATED IN UMAP SPACE
# CENTERS ARE IN UMAP SPACE AND NOT SIGLIP SPACE
# OR
# AVERAGE EMBEDDING IS TOO SPARSE AND DOESN'T CAPTURE THE TERMS

CLUSTER = 1

cemb, cimgs = avg_cluster_embedding[CLUSTER], by_clust_dist[CLUSTER]
iidx = 2

img = PImage.open(f"./herbario-media/imgs/arts/500/{cimgs[iidx]}.jpg")
img_np = np.array(img)

similarity_map_np = msl.get_gradient_activation_map(img, cemb, text_idx=None)
# similarity_map = ((1e4 * similarity_map_np).astype(int).astype(float) / 1e4).tolist()

masked_img = mask_image(img, similarity_map_np)
heatmap_img = heatmap_image(similarity_map_np, size=img.size)

print(cimgs[iidx])
display(img)
display(masked_img)
display(PImage.blend(img, heatmap_img, 0.65))